> **対応するブログ記事**: [#6 Perseusなしで！Pythonでプロテオミクスデータを前処理する](../blog/article-06-preprocess.md)
>
> このNotebookはブログ記事 #6 のコードをセルごとに実行できるインタラクティブ版です。Log2変換・欠損値フィルタ・downshift補完の詳しい解説はブログ記事を参照してください。

# Step 6: データ前処理

Sage が出力したタンパク質定量マトリクスを統計解析に適した形に整える。
処理: **Log2変換** → **有効値フィルタリング（70%）** → **欠損値補完（downshift法）**

In [ ]:
import os       # ファイルパスの結合やディレクトリ操作に使用する標準ライブラリ
import numpy as np   # 数値計算ライブラリ（log2変換・正規乱数生成などに使用）
import pandas as pd  # データフレーム操作ライブラリ（CSV読み書き・データ加工に使用）

In [ ]:
# --- パス設定 ---
# 結果ファイルを格納するディレクトリ（notebooks/から見た相対パス）
RESULTS_DIR = "../results"
# Sage（Step 4）が出力したタンパク質定量マトリクスCSVのパスを組み立てる
INPUT_CSV = os.path.join(RESULTS_DIR, "protein_matrix_from_sage.csv")

# --- Perseus互換パラメータ ---
# downshift: 有効値の平均から何SD（標準偏差）下に補完値の中心を置くか
# 値が大きいほど補完値は低くなる（検出限界以下の微量タンパク質を模擬）
DOWNSHIFT = 2.4
# width: 補完値の分布幅を元のSDの何倍にするか
# 小さいほど補完値のばらつきが少なくなる
WIDTH = 0.3
# 少なくとも1群（Normal or Tumor）でこの割合以上の有効値が必要
# 0.70 = 70%以上のサンプルで検出されていないタンパク質は除去する
VALID_RATIO = 0.70

## 1. データ読み込み

In [ ]:
# CSVファイルを読み込み、最初の列（タンパク質名）をインデックスに設定する
df = pd.read_csv(INPUT_CSV, index_col=0)
# インデックス列の名前を "Protein" に統一する（後続処理で参照しやすくするため）
df.index.name = "Protein"

# サンプル名の末尾に "-N" を含むカラムをNormal群として抽出する
normal_samples = [c for c in df.columns if "-N" in c]
# サンプル名の末尾に "-T" を含むカラムをTumor群として抽出する
tumor_samples  = [c for c in df.columns if "-T" in c]
# 群名とサンプルリストの対応辞書（フィルタリング時に群ごとの処理に使う）
groups = {"Normal": normal_samples, "Tumor": tumor_samples}

# 読み込んだデータの概要を表示する（行数=タンパク質数、列数=サンプル数）
print(f"読み込み: {df.shape[0]} タンパク質 × {df.shape[1]} サンプル")
# 各群のサンプル数を確認する
print(f"  Normal: {len(normal_samples)}, Tumor: {len(tumor_samples)}")

## 2. Log2変換

質量分析の強度値は 10^6〜10^9 と桁が大きく分布が偏る。
Log2変換で正規分布に近づけ、差が「倍率」に対応するようになる（差1 = 2倍変化）。

In [ ]:
# 全サンプルの中央値の中央値を計算し、データのスケールを判定する
# 生の強度値なら10^6〜10^9程度、log2済みなら20〜30程度になる
median_val = df.median().median()

# 中央値が100より大きければ生データと判断してLog2変換を行う
if median_val > 100:
    # 0値はNaN（欠損）に変換してからLog2をとる
    # （0のlog2は-∞になるため、0は「検出されなかった」として欠損扱いにする）
    df = np.log2(df.replace(0, np.nan))
    # 変換前の中央値を表示して、変換が実行されたことを確認する
    print(f"Log2変換実行（変換前の中央値: {median_val:.1f}）")
else:
    # 既にlog2スケールの場合はスキップする（二重変換を防止）
    print(f"既にlog2スケール（中央値: {median_val:.1f}）→ スキップ")

## 3. 有効値フィルタリング

ほとんどのサンプルで検出されないタンパク質は統計的に信頼できない。
少なくとも1群で70%以上のサンプルに有効値があるタンパク質のみ残す。

In [ ]:
def filter_by_valid_ratio(df, groups, ratio=VALID_RATIO):
    """いずれかの群で有効値割合 >= ratio を満たすタンパク質を残す。"""
    # 全タンパク質をFalse（除去対象）で初期化する
    keep = pd.Series(False, index=df.index)
    # 各群（Normal / Tumor）について有効値の割合を計算する
    for samples in groups.values():
        # 各タンパク質の有効値（NaNでない値）の数をサンプル数で割って割合を求める
        valid = df[samples].notna().sum(axis=1) / len(samples)
        # いずれかの群で閾値以上であればTrueにする（OR条件）
        keep |= (valid >= ratio)
    # keep=Trueのタンパク質のみ返す
    return df[keep]

# フィルタリング前のタンパク質数を記録する（除去数の計算用）
n_before = len(df)
# 有効値割合フィルタを適用して信頼性の低いタンパク質を除去する
df = filter_by_valid_ratio(df, groups)
# フィルタリング結果を表示する（前後の数と除去された数）
print(f"フィルタリング: {n_before} → {len(df)} タンパク質（{n_before - len(df)} 除去）")

## 4. 欠損値補完（downshift法）

質量分析で「検出されなかった」タンパク質は、発現量が検出限界以下だったと考える。
各サンプルの有効値分布から downshift × SD 分だけ低い位置に正規分布を設定し、
そこからランダムサンプリングして欠損を埋める（Perseus互換）。

In [ ]:
def impute_downshift(df, downshift=DOWNSHIFT, width=WIDTH):
    """Perseus互換: 各サンプルごとに低値側から欠損を補完する。"""
    # 元のデータを変更しないようコピーを作成する
    df = df.copy()
    # 補完された値の総数をカウントする変数
    total = 0
    # 各サンプル（カラム）ごとに個別に補完処理を行う
    for col in df.columns:
        # 現在のサンプルの有効値（NaNでない値）のみ取り出す
        valid = df[col].dropna()
        # 有効値が0個の場合はスキップする（平均・SDが計算できないため）
        if len(valid) == 0:
            continue
        # 補完値の中心 = 有効値の平均 - downshift × 有効値のSD
        # 有効値の分布より低い位置に中心を設定する（検出限界以下を模擬）
        imp_mean = valid.mean() - downshift * valid.std()
        # 補完値の標準偏差 = width × 有効値のSD
        # 元の分布より狭い範囲にばらつかせる
        imp_std  = width * valid.std()
        # 現在のサンプルで欠損（NaN）となっている行のブールマスクを取得する
        mask = df[col].isna()
        # 欠損値の個数を数える
        n = mask.sum()
        # 欠損がある場合のみ補完を実行する
        if n > 0:
            # 正規分布N(imp_mean, imp_std)からn個の乱数を生成して欠損箇所に代入する
            df.loc[mask, col] = np.random.normal(imp_mean, imp_std, n)
            # 補完した値の数を累計する
            total += n
    # 補完済みデータフレームと補完値の総数を返す
    return df, total

# 乱数のシードを固定して再現性を確保する（毎回同じ補完値が生成される）
np.random.seed(42)
# downshift法で欠損値を補完する
df, n_imputed = impute_downshift(df)
# 補完結果を表示する（補完した値の数と使用したパラメータ）
print(f"欠損値補完: {n_imputed} 値（downshift={DOWNSHIFT}, width={WIDTH}）")
# 補完後に残っている欠損値の数を確認する（正常なら0になるはず）
print(f"残り欠損: {df.isna().sum().sum()}")

## 5. 保存

In [ ]:
# 前処理済みデータをCSVとして保存する（後続の可視化・統計解析で使用）
df.to_csv(os.path.join(RESULTS_DIR, "preprocessed_data.csv"))

# サンプル情報テーブルを作成する（各サンプルがどの群に属するかを記録）
# Normal群のサンプル名とTumor群のサンプル名を縦に結合する
sample_info = pd.DataFrame({
    "Sample": normal_samples + tumor_samples,  # 全サンプル名のリスト
    # 対応するConditionラベルを同じ順序で付与する
    "Condition": ["Normal"] * len(normal_samples) + ["Tumor"] * len(tumor_samples)
})
# サンプル情報をCSVとして保存する（index=Falseで行番号を含めない）
sample_info.to_csv(os.path.join(RESULTS_DIR, "sample_info.csv"), index=False)

# 保存結果の概要を表示する
print(f"保存完了: {df.shape[0]} タンパク質 × {df.shape[1]} サンプル")
print(f"  → preprocessed_data.csv")   # 前処理済みデータの出力先
print(f"  → sample_info.csv")          # サンプル情報の出力先